# 02 — Enzyme–Protein Screening
This notebook applies the reusable HERALD screening workflow to compare all currently defined whey proteins against all currently defined enzymes.

The goal is to identify protein–enzyme combinations that produce peptide profiles with higher exploratory AMP-like scores.

Unlike the first notebook, which built the workflow step by step for a single example, this notebook now uses the reusable screen_combinations() function from screening.py. This keeps the notebook focused on running, inspecting, ranking, and saving the screening results rather than duplicating project logic.

## Import HERALD Screening Workflow

This section imports the reusable screening function from the HERALD package.

The screen_combinations() function combines several project modules:

* proteins.py retrieves protein sequences.
* digestion.py simulates enzymatic digestion.
* scoring.py computes peptide features and simple AMP-like scores.
* screening.py summarizes results across protein–enzyme combinations.

Using the reusable function keeps the notebook concise and ensures that the same screening logic can be used in future notebooks, scripts, or analysis workflows.

In [8]:
import sys
import os
import pandas as pd

PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

from herald.proteins import protein_sequence, WHEY_PROTEINS
from herald.enzymes import ENZYME_RULES
from herald.digestion import digest_sequence
from herald.scoring import peptide_features
from herald.screening import screen_combinations

## Define Screening Scope
The default screen uses the current HERALD protein and enzyme definitions:

* Proteins are defined in WHEY_PROTEINS.
* Enzymes and cleavage rules are defined in ENZYME_RULES.

At this stage, the screen is intentionally small and exploratory. Additional whey proteins, food-grade proteases, and experimentally realistic digestion conditions can be added later as the project expands.

In [9]:
print("Proteins:")
for protein_name, accession_id in WHEY_PROTEINS.items():
    print(f"- {protein_name}: {accession_id}")

print("\nEnzymes:")
for enzyme_name in ENZYME_RULES.keys():
    print(f"- {enzyme_name}")

Proteins:
- beta-lactoglobulin: P02754
- alpha-lactalbumin: P00709
- lactoferrin: P24627
- BSA: P02769

Enzymes:
- trypsin
- chymotrypsin


## Run Protein–Enzyme Screening

The screening workflow evaluates each protein–enzyme combination using the current HERALD pipeline.

For each combination, the workflow:

1. retrieves the protein sequence,
2. simulates enzymatic digestion,
3. computes peptide-level AMP-like features,
4. identifies the top-scoring peptide, and
5. summarizes enzyme-level screening metrics.

The resulting table provides one row per protein–enzyme combination.

In [10]:
results_df = screen_combinations()
results_df

,protein,accession_id,enzyme,min_length,max_length,num_peptides,max_score,top_peptide,num_score_ge_3,num_score_ge_4,avg_score
0,beta-lactoglobulin,P02754,trypsin,4,50,13,3,CLLLALALTCGAQALIVTQTMK,7,0,2.307692
1,beta-lactoglobulin,P02754,chymotrypsin,4,50,19,4,ENGECAQKKIIAEKTKIPAVF,4,2,1.578947
2,alpha-lactalbumin,P00709,trypsin,4,50,12,4,GIDYWLAHK,5,1,2.250000
3,alpha-lactalbumin,P00709,chymotrypsin,4,50,16,3,AKQF,3,0,1.625000
4,lactoferrin,P24627,trypsin,4,50,58,5,QVLLHQQALFGK,34,9,2.465517
5,lactoferrin,P24627,chymotrypsin,4,50,67,5,AAPRKNVRW,26,7,2.134328
6,BSA,P02769,trypsin,4,50,62,4,WVTFISLLLLFSSAYSR,21,3,1.951613
7,BSA,P02769,chymotrypsin,4,50,61,5,EIARRHPY,17,5,1.901639


## Rank Screening Results

The screening results are sorted to prioritize combinations that produce more high-scoring peptides.

The current ranking uses the following exploratory criteria:

1. number of peptides with `simple_amp_score ≥ 4`,
2. number of peptides with `simple_amp_score ≥ 3`,
3. maximum simple AMP-like score,
4. average simple AMP-like score.

These rankings are based on a transparent rule-based scoring baseline. They should be interpreted as exploratory prioritization results, not validated antimicrobial activity predictions.

In [11]:
ranked_results_df = results_df.sort_values(
    by=["num_score_ge_4", "num_score_ge_3", "max_score", "avg_score"],
    ascending=False,
)

ranked_results_df

,protein,accession_id,enzyme,min_length,max_length,num_peptides,max_score,top_peptide,num_score_ge_3,num_score_ge_4,avg_score
4,lactoferrin,P24627,trypsin,4,50,58,5,QVLLHQQALFGK,34,9,2.465517
5,lactoferrin,P24627,chymotrypsin,4,50,67,5,AAPRKNVRW,26,7,2.134328
7,BSA,P02769,chymotrypsin,4,50,61,5,EIARRHPY,17,5,1.901639
6,BSA,P02769,trypsin,4,50,62,4,WVTFISLLLLFSSAYSR,21,3,1.951613
1,beta-lactoglobulin,P02754,chymotrypsin,4,50,19,4,ENGECAQKKIIAEKTKIPAVF,4,2,1.578947
2,alpha-lactalbumin,P00709,trypsin,4,50,12,4,GIDYWLAHK,5,1,2.250000
0,beta-lactoglobulin,P02754,trypsin,4,50,13,3,CLLLALALTCGAQALIVTQTMK,7,0,2.307692
3,alpha-lactalbumin,P00709,chymotrypsin,4,50,16,3,AKQF,3,0,1.625000


## Save Ranked Screening Results

The ranked enzyme–protein screening table is saved as a CSV file in `data/processed/`.

This creates a reusable output that can be inspected later, shared with collaborators, or used as input for downstream analysis.

In [51]:
PROJECT_ROOT = os.path.abspath("..")
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

output_path = os.path.join(
    PROCESSED_DATA_DIR,
    "enzyme_protein_screening_results.csv",
)

ranked_results_df.to_csv(output_path, index=False)

print(f"Saved results to: {output_path}")

Saved results to: /Users/lukas/Developer/herald/data/processed/enzyme_protein_screening_results.csv
